## Portfolio Analysis Agents - Advanced Version

This notebook demonstrates creating advanced Foundry agents for portfolio analysis with multi-turn conversations and detailed analytics.

### Installing Required Libraries

In [ ]:
%pip install azure-ai-projects==2.0.0b2 openai==1.109.1 python-dotenv azure-identity yfinance requests pandas numpy

### Importing Required Libraries

In [ ]:
import os
import json
import numpy as np
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
import yfinance as yf
import pandas as pd
import requests
from datetime import datetime, timedelta

load_dotenv()

### Setup Foundry Client

In [ ]:
foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")

project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential(),
)

openai_client = project_client.get_openai_client()
print("Foundry client initialized")

### Define Client Portfolio Data

In [ ]:
client_portfolios = {
    "Client_1": {
        "name": "Client 1",
        "stocks": ["TCS.NS", "WIPRO.NS", "INFY.NS", "HDFCBANK.NS"],
        "mutual_funds": ["119551", "119022"]
    },
    "Client_2": {
        "name": "Client 2",
        "stocks": ["RELIANCE.NS", "HDFCBANK.NS", "ICICIBANK.NS", "LT.NS"],
        "mutual_funds": ["119551", "120485"]
    },
    "Client_3": {
        "name": "Client 3",
        "stocks": ["TATAMOTORS.NS", "ADANIGREEN.NS", "ETERNAL.NS", "PAYTM.NS"],
        "mutual_funds": ["119598", "119618"]
    },
    "Client_4": {
        "name": "Client 4",
        "stocks": ["ITC.NS", "COALINDIA.NS", "NTPC.NS", "POWERGRID.NS"],
        "mutual_funds": ["119547"]
    },
    "Client_5": {
        "name": "Client 5",
        "stocks": ["INFY.NS", "SUNPHARMA.NS", "BHARTIARTL.NS", "TATACONSUM.NS"],
        "mutual_funds": ["119510", "120303"]
    }
}

print(f"Loaded portfolios for {len(client_portfolios)} clients")

### Data Fetching Functions

In [ ]:
def get_stock_price_history(symbol, days=30):
    try:
        end_date = datetime.today()
        start_date = end_date - timedelta(days=days)
        
        df = yf.download(symbol, start=start_date, end=end_date, auto_adjust=True, progress=False)
        
        if df.empty:
            return {"error": "No data available"}
        
        current_price = df['Close'].iloc[-1]
        avg_price = df['Close'].mean()
        high_52week = df['High'].max()
        low_52week = df['Low'].min()
        price_change = ((current_price - df['Close'].iloc[0]) / df['Close'].iloc[0]) * 100
        
        return {
            "ticker": symbol,
            "current_price": round(current_price, 2),
            "30_day_average": round(avg_price, 2),
            "52_week_high": round(high_52week, 2),
            "52_week_low": round(low_52week, 2),
            "30_day_change_percent": round(price_change, 2),
            "trading_volume_avg": int(df['Volume'].mean())
        }
    except Exception as e:
        return {"ticker": symbol, "error": str(e)}

print("Data functions ready")

### Get Sector Analysis

In [ ]:
def get_sector_analysis(client_id):
    sector_map = {
        "TCS.NS": "IT",
        "WIPRO.NS": "IT",
        "INFY.NS": "IT",
        "HDFCBANK.NS": "Banking",
        "RELIANCE.NS": "Energy",
        "ICICIBANK.NS": "Banking",
        "LT.NS": "Engineering",
        "TATAMOTORS.NS": "Automotive",
        "ADANIGREEN.NS": "Energy",
        "ETERNAL.NS": "E-commerce",
        "PAYTM.NS": "Fintech",
        "ITC.NS": "FMCG",
        "COALINDIA.NS": "Energy",
        "NTPC.NS": "Energy",
        "POWERGRID.NS": "Infrastructure",
        "SUNPHARMA.NS": "Pharma",
        "BHARTIARTL.NS": "Telecom",
        "TATACONSUM.NS": "FMCG"
    }
    
    portfolio = client_portfolios.get(client_id)
    if not portfolio:
        return {"error": "Client not found"}
    
    sectors = {}
    for stock in portfolio['stocks']:
        sector = sector_map.get(stock, "Other")
        if sector not in sectors:
            sectors[sector] = []
        sectors[sector].append(stock)
    
    return {
        "client_id": client_id,
        "sectors": sectors,
        "sector_breakdown": {sector: f"{(len(stocks)/len(portfolio['stocks'])*100):.1f}%" for sector, stocks in sectors.items()}
    }

print("Sector analysis ready")

### Create Advanced Portfolio Agents

In [ ]:
def create_agent_instructions(client_id):
    portfolio = client_portfolios.get(client_id)
    sector_analysis = get_sector_analysis(client_id)
    
    if "error" in sector_analysis:
        return None
    
    sectors_text = ", ".join([f"{sector} ({', '.join(stocks)})" for sector, stocks in sector_analysis['sectors'].items()])
    
    instructions = f"""You are an advanced portfolio analyst for {portfolio['name']}.

PORTFOLIO OVERVIEW:
Holdings: {', '.join(portfolio['stocks'])}
Mutual Funds: {len(portfolio['mutual_funds'])} funds

Sector Exposure: {sectors_text}

YOUR RESPONSIBILITIES:
1. Provide real-time stock analysis and technical metrics
2. Assess portfolio diversification and risk exposure
3. Give insights on security performance
4. Recommend rebalancing strategies
5. Answer market trend questions

Use Yahoo Finance data to provide accurate information."""
    
    return instructions

advanced_agents = {}

for client_id, portfolio in client_portfolios.items():
    agent_name = f"portfolio-analyst-{client_id.lower()}"
    instructions = create_agent_instructions(client_id)
    
    if instructions:
        agent = project_client.agents.create_version(
            agent_name=agent_name,
            definition=PromptAgentDefinition(
                model=model_deployment_name,
                instructions=instructions
            )
        )
        
        advanced_agents[client_id] = {
            "agent": agent,
            "agent_name": agent_name,
            "portfolio": portfolio
        }
        
        print(f"Created: {agent_name}")

print(f"Total agents created: {len(advanced_agents)}")

### Create Conversations

In [ ]:
conversations = {}

for client_id in advanced_agents.keys():
    conversation = openai_client.conversations.create()
    conversations[client_id] = conversation.id
    print(f"Conversation created for {client_id}")

print(f"Total conversations: {len(conversations)}")

### Query Agent - Client 1

In [ ]:
client_id = "Client_1"
query = "What is the current status of my IT sector holdings?"

response = openai_client.responses.create(
    conversation=conversations[client_id],
    extra_body={
        "agent": {
            "name": advanced_agents[client_id]["agent_name"],
            "type": "agent_reference"
        }
    },
    input=query
)

print(f"Query: {query}")
print(f"Response:\n{response.output_text}")

### Query Agent - Client 2

In [ ]:
client_id = "Client_2"
query = "How does my portfolio compare to the Nifty 50 index?"

response = openai_client.responses.create(
    conversation=conversations[client_id],
    extra_body={
        "agent": {
            "name": advanced_agents[client_id]["agent_name"],
            "type": "agent_reference"
        }
    },
    input=query
)

print(f"Query: {query}")
print(f"Response:\n{response.output_text}")

### Summary

In [ ]:
print("="*60)
print("ADVANCED PORTFOLIO AGENTS - SUMMARY")
print("="*60)

for client_id, agent_info in advanced_agents.items():
    print(f"\n{agent_info['portfolio']['name']}")
    print(f"  Agent: {agent_info['agent_name']}")
    print(f"  ID: {agent_info['agent']['id']}")
    print(f"  Stocks: {len(agent_info['portfolio']['stocks'])}")
    print(f"  Funds: {len(agent_info['portfolio']['mutual_funds'])}")

print(f"\n{'='*60}")
print(f"Total Agents: {len(advanced_agents)}")
print(f"Total Conversations: {len(conversations)}")
print(f"{'='*60}")